# Yeast promoter design with TACO on Google Colab, using PyTorch

This notebook runs the published TACO code<sup>1</sup> on yeast promoters on Google Colab, with the three seeds 0, 1 and 42. It is a modification of the published work, and my own variant with the writer and the two scoring models coming from the regLM yeast files, and the motif weight is 0 (the motif files from TACO are not used).

It is only handling **Part 1** of the total analysis, in short:

1. Steps 1 to 4: imports, settings, graphics card, Google Drive.
2. Steps 5 to 7: unpack the TACO code, download only the three model files the run uses, build the 200 weak starting sequences.
3. Steps 8 to 13: build a separate Python 3.9 environment named `taco_env`, install the packages, clone the HyenaDNA code, change the file paths in a working copy of TACO.
4. Steps 14 to 16: a tiny test run, the three full runs (100 rounds each), and the summary table.

### **Step 1: Package Import**

In [ ]:
from importlib.metadata import version, PackageNotFoundError
import pathlib
import platform
import shutil
import subprocess
import tarfile
import zipfile

import pandas
import requests

# Only exists on Colab. Outside Colab, the name stays empty.
try:
    from google.colab import drive
except ImportError:
    drive = None

print("Python==" + platform.python_version())

for package in ["pandas", "requests"]:
    try:
        print(package + "==" + version(package))
    except PackageNotFoundError:
        print(package + ": not installed yet")

### **Step 2: Settings**

All names and folders are written down in this one cell. If a name changes, it is changed here only.

In [ ]:
# This is the only cell that needs future editing.

# The folder in my Google Drive that holds all big files.
drive_dir = "Reinforcement_Learning_for_Promoter_Design"

# The subfolder of that Drive folder, and the name of the zip file inside it.
code_subfolder = "taco_code"
taco_zip_file = "TACO-main.zip"

# The working folder for this session. Colab erases this folder when the session ends.
working_folder = "/content/working"

# The separate Python 3.9 environment for TACO.
environment_folder = "/content/taco_env"

# Leave this empty on Colab. Only for running the notebook outside Colab: a local folder that stands in for the Drive folder.
local_dir = ""

# The regLM yeast archive on Zenodo, and the three files inside it that the run uses (writer, reward model, checker model).
yeast_archive_address = "https://zenodo.org/records/12668907/files/yeast_promoters.tar.gz?download=1"
needed_model_files = [
    "yeast_promoters/04_reglm/yeast_reglm.ckpt",
    "yeast_promoters/02_regression_paired/yeast_regression_paired_complex.ckpt",
    "yeast_promoters/03_regression_separate/yeast_regression_separate_complex.ckpt",
]

# The native yeast promoters, and how the 200 weak starting sequences are made from them.
native_complex_address = "https://zenodo.org/records/4436477/files/Native_complex.csv"
flank_length_start = 17
flank_length_end = 13
promoter_length = 80
weakest_bin_upper_limit = 5.391858
starting_sequence_count = 200
starting_sequence_seed = 0

# The FlashAttention wheel (a ready-made install file) that fits Python 3.9, torch 2.1 and CUDA 12.2.
flash_attention_wheel_address = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.9.post1/"
    "flash_attn-2.5.9.post1+cu122torch2.1cxx11abiFALSE-cp39-cp39-linux_x86_64.whl"
)

# The TACO runs: tiny test, then the full runs.
tiny_test_rounds = 2
tiny_test_batch_size = 8
full_run_rounds = 100
full_run_batch_size = 256
seeds = [0, 1, 42]

### **Step 3: Check the graphics card**

TACO is written for a graphics card from NVIDIA, so it cannot run without one. The command `nvidia-smi` asks the computer which card it has.

In [ ]:
# HELPER FUNCTION
# Asking the computer for its graphics card and printing the answer.
def show_graphics_card_information():
    try:
        completed_process = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        print(completed_process.stdout)
    except FileNotFoundError:
        print("No graphics card found. On Colab: Runtime, Change runtime type, choose a graphics card.")

show_graphics_card_information()

### **Step 4: Connect Google Drive**

Colab erases its own files when the session ends, but Drive keeps them. So the zip, the downloaded model files and the results all live in the Drive folder.

In [ ]:
# HELPER FUNCTION
# Connecting Drive on Colab, and returning the project folder in it.
def find_drive_dir():
    if local_dir != "":
        # Only used outside Colab.
        return pathlib.Path(local_dir)
    if drive is None:
        raise RuntimeError("Not running on Colab. Fill in local_dir in Step 2.")
    drive.mount("/content/drive")
    return pathlib.Path("/content/drive/MyDrive") / drive_dir

drive_dir_name = find_drive_dir()
print("Drive project folder:", drive_dir_name)
print("Folder exists:", drive_dir_name.exists())

### **Step 5: Unpack the TACO zip into the working folder**

The zip is copied from Drive and unpacked in the working folder.

Working from the session's own disk is faster than working from Drive, and the zip in Drive stays untouched. If the working folder already exists, it is deleted first, so that old changes are never mixed in.

In [ ]:
taco_zip_path = drive_dir_name / code_subfolder / taco_zip_file
working_folder_path = pathlib.Path(working_folder)

# Stopping early, with a clear message, if the zip is not where Step 2 says.
if not taco_zip_path.exists():
    raise FileNotFoundError("The zip was not found at " + str(taco_zip_path) + ". Check the names in Step 2.")

if working_folder_path.exists():
    shutil.rmtree(working_folder_path)
working_folder_path.mkdir(parents=True)

with zipfile.ZipFile(taco_zip_path) as zip_file:
    zip_file.extractall(working_folder_path)
    print("Files unpacked:", len(zip_file.namelist()))

taco_folder = working_folder_path / "TACO-main"
print("Unpacked folder:", taco_folder)

# The four files that the run uses and that are changed in Step 11.
for relative_file_name in ["reinforce_mbo.py", "dna_optimizers/mbo_optimizaer.py", "reglm_src/reglm/lightning.py", "reglm_src/reglm/regression.py"]:
    print(relative_file_name, (taco_folder / relative_file_name).exists())

### **Step 6: Download the 3 model files that the run uses**

The regLM yeast archive on Zenodo is 270.5 megabytes and holds six model files. The run uses three of them: the **writer** (the model that writes promoters, which TACO trains further), the **reward model** (the regLM `paired` model, which scores the writer's sequences during training) and the **checker model** (the regLM `separate` model, which TACO uses to evaluate). The archive is read as a stream, and only these three files are saved, in the `model_files` folder in Drive.

In [ ]:
# HELPER FUNCTION
# Reading the archive from Zenodo as a stream and saving only the files that are listed in Step 2.
def download_needed_model_files(model_files_folder):
    files_to_find = set(needed_model_files)
    with requests.get(yeast_archive_address, stream=True, timeout=120) as zenodo_download:
        zenodo_download.raise_for_status()
        with tarfile.open(fileobj=zenodo_download.raw, mode="r|gz") as archive:
            for member in archive:
                if member.name in files_to_find:
                    # filter="data" refuses unsafe file paths inside the archive.
                    archive.extract(member, path=model_files_folder, filter="data")
                    files_to_find.discard(member.name)
                    print("Saved:", member.name, "|", round(member.size / 1e6, 1), "megabytes")
                    if not files_to_find:
                        break
    if files_to_find:
        raise FileNotFoundError("Not found in the archive: " + str(sorted(files_to_find)))

model_files_folder = drive_dir_name / "model_files"
model_files_folder.mkdir(parents=True, exist_ok=True)
download_needed_model_files(model_files_folder)

for model_file_name in needed_model_files:
    model_file = model_files_folder / model_file_name
    print(model_file.relative_to(drive_dir_name), "|", round(model_file.stat().st_size / 1e6, 1), "megabytes")

### **Step 7: Build the starting sequences for the yeast task**

TACO starts from a set of weak promoters and tries to make them stronger. The regLM authors sampled 200 weak sequences for their benchmark. Here the same is done with the native yeast promoters (Zenodo record 4436477).

The steps follow the regLM notebook `01_process.ipynb`: remove the flanking letters (17 at the start, 13 at the end), drop sequences that contain `N`, keep only sequences that are 80 letters long, choose the weak ones (below the limit 5.391858, the lowest of the five activity groups in regLM's `02_categorize.ipynb`) and sample 200 of them with a fixed seed.

In [ ]:
starting_sequences_folder = drive_dir_name / "data" / "complex"
starting_sequences_folder.mkdir(parents=True, exist_ok=True)
starting_sequences_file = starting_sequences_folder / "mbo.csv"

# The first row is skipped because the column names are given here.
native_complex = pandas.read_csv(native_complex_address, usecols=(0, 1, 2), skiprows=1, names=["index", "sequence", "expression"])
print("Native promoters read:", len(native_complex))

# Removing the flanking letters, the sequences with N, and the sequences that are not 80 letters long.
native_complex["sequence"] = native_complex["sequence"].str[flank_length_start:-flank_length_end]
native_complex = native_complex[~native_complex["sequence"].str.contains("N")]
native_complex = native_complex[native_complex["sequence"].str.len() == promoter_length]
print("Left after the filters:", len(native_complex))

# Keeping the weak promoters, and sampling from them with a fixed seed.
weak_native = native_complex[native_complex["expression"] < weakest_bin_upper_limit]
print("Weak promoters:", len(weak_native))

starting_sequences = weak_native.sample(starting_sequence_count, random_state=starting_sequence_seed)[["sequence", "expression"]]
starting_sequences.columns = ["sequence", "target"]
starting_sequences.to_csv(starting_sequences_file, index=False)
print("Saved:", starting_sequences_file, "| rows:", len(starting_sequences))

### **Step 8: Make the Python 3.9 environment `taco_env` and install all its packages**

TACO needs older packages than Colab has, so everything it needs goes into a separate Python 3.9 environment named `taco_env`, made with the tool `uv`. All installs are in this one cell, in the order in which they must run. The notebook itself stays on Colab's Python, and TACO is run with the Python of `taco_env`.

1. A **constraints file** is written first. It fixes torch, torchvision, torchaudio and numpy, and every later install obeys it. An earlier version of this notebook lost time because one package silently upgraded torch to 2.8.0 and broke FlashAttention, and numpy 2 is too new for this code.
2. **Torch 2.1.0**, torchvision 0.16.0 and torchaudio 2.1.0 from the CUDA 12.1 package index, then numpy below 2.
3. **FlashAttention 2.5.9.post1** from a ready-made wheel (an install file that needs no compiling), without its own dependencies (`--no-deps`).
4. **All other packages**, in one command, under the constraints file. The pins come from what worked in the earlier runs: `transformers` 4.55.4, `enformer_pytorch` 0.8.10 and `vector-quantize-pytorch` 1.25.2 (the newer versions of the last two pulled in torch 2.8.0). `setuptools` is pinned below 70 because the old `pytorch_lightning` needs its `pkg_resources` module. The motif package `pymemesuite` is imported by the code even with a motif weight of 0, so it must install.
5. **`pytorch_lightning` 1.8.6** without its dependencies (`--no-deps`), as TACO's own install script does.

Packages without a pin take the newest version that fits Python 3.9, which is why Step 14 saves the exact versions at the end.

In [ ]:
%%time
environment_folder_path = pathlib.Path(environment_folder)
environment_python = environment_folder_path / "bin" / "python"

# Starting from a clean environment folder.
if environment_folder_path.exists():
    shutil.rmtree(environment_folder_path)

# The constraints file that every later install obeys.
constraints_file = pathlib.Path("/content/constraints.txt")
constraints_file.write_text("torch==2.1.0\ntorchvision==0.16.0\ntorchaudio==2.1.0\nnumpy<2\n")

!pip install -q uv
!uv venv --python 3.9 {environment_folder}
!{environment_python} --version
!uv pip install --python {environment_python} torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python {environment_python} -c {constraints_file} "numpy<2"
!{environment_python} -c "import torch; print('torch', torch.__version__, '| graphics card available:', torch.cuda.is_available())"
!uv pip install --python {environment_python} -c {constraints_file} --no-deps {flash_attention_wheel_address}
!uv pip install --python {environment_python} -c {constraints_file} pandas scipy scikit-learn tqdm pyyaml packaging fsspec torchmetrics lightning_utilities tensorboardX transformers==4.55.4 wandb anndata enformer_pytorch==0.8.10 vector-quantize-pytorch==1.25.2 pymemesuite einops omegaconf hydra-core rich opt_einsum "setuptools<70"
!uv pip install --python {environment_python} -c {constraints_file} --no-deps pytorch_lightning==1.8.6

### **Step 9: Check the imports**

The same imports that TACO's model code makes, done once before the long steps. **Why:** a missing package shows up here in seconds and not in the middle of a run. This only shows that the code loads; the tiny run in Step 14 shows that it works.

In [ ]:
!uv pip list --python {environment_python} | grep -i -E "torch|numpy|flash|triton|enformer|transformers"
!{environment_python} -c "from flash_attn.modules.mha import MHA, ParallelMHA; from flash_attn.modules.mlp import Mlp, FusedMLP, ParallelFusedMLP; from flash_attn.modules.block import Block; from flash_attn.modules.embedding import GPT2Embeddings, ParallelGPT2Embeddings; from flash_attn.utils.generation import GenerationMixin; from flash_attn.utils.distributed import sync_shared_params, all_gather_raw; print('FlashAttention imports work')"
!{environment_python} -c "import pkg_resources, pytorch_lightning; print('pytorch_lightning', pytorch_lightning.__version__, 'imports work')"

### **Step 10: Clone the official HyenaDNA code**

TACO's model code imports from a package called `src`, which is the official HyenaDNA repository. It is cloned without its history (`--depth 1`). The line at the end prints which version of that repository was cloned.

The clone has no fixed version, so the printed identifier is the record of what was used. Copy it into the README.

In [ ]:
hyena_code_folder = working_folder_path / "hyena-dna"

In [ ]:
!git clone --depth 1 https://github.com/HazyResearch/hyena-dna.git {hyena_code_folder}
!git -C {hyena_code_folder} rev-parse HEAD
print((hyena_code_folder / "src" / "models" / "sequence" / "long_conv_lm.py").exists())

### **Step 11: Point the TACO code to the real files**

The `/blob/...` paths in the TACO code belong to the authors' computer. In the working copy only (the zip stays untouched), they are changed to the files in Drive:
- the writer, in `reinforce_mbo.py`;
- the reward model (the regLM `paired` model) and the checker model (the regLM `separate` model), in `mbo_optimizaer.py`;
- the two HyenaDNA locations, in `lightning.py`.

Three more changes:
- the motif files are not loaded when the motif weight is 0, because they are then never used;
- two speed-up options are switched off (`fused_dropout_add_ln` and `fused_mlp`), because they need separately compiled FlashAttention parts that are not installed. This gives the same calculation, only without the combined fast step;
- the starting sequences from Step 7 are copied to the place that TACO reads.

Each change is made by a helper function that refuses to continue unless the old text is found exactly once, so a wrong change never goes unnoticed. If a change was already made, it says so.

In [ ]:
reglm_model_folder = drive_dir_name / "model_files"
writer_file = reglm_model_folder / needed_model_files[0]
reward_model_file = reglm_model_folder / needed_model_files[1]
checker_model_file = reglm_model_folder / needed_model_files[2]
hyena_base_model_folder = working_folder_path / "hyenadna-medium-160k-seqlen"

# HELPER FUNCTION
# Replacing one exact piece of text in a file. It refuses to continue if the text is not found exactly once.
def replace_text_in_file(file_path, old_text, new_text):
    file_text = file_path.read_text()
    if new_text in file_text:
        print("Already patched:", file_path.name)
        return
    if file_text.count(old_text) != 1:
        raise ValueError("Expected exactly one match in " + file_path.name + " but found " + str(file_text.count(old_text)))
    file_path.write_text(file_text.replace(old_text, new_text))
    print("Patched:", file_path.name)

optimizer_file = taco_folder / "dna_optimizers" / "mbo_optimizaer.py"
replace_text_in_file(optimizer_file,
    '"/blob/Data/DNA/Design/guide_model/yeast_regression_paired_complex_offline.ckpt"',
    '"' + str(reward_model_file) + '"')
replace_text_in_file(optimizer_file,
    '"/blob/ICLR/oracle/yeast_regression_paired_complex.ckpt"',
    '"' + str(checker_model_file) + '"')

replace_text_in_file(taco_folder / "reinforce_mbo.py",
    '"/blob/ICLR/reglm/yeast_reglm.ckpt"',
    '"' + str(writer_file) + '"')

lightning_file = taco_folder / "reglm_src" / "reglm" / "lightning.py"
replace_text_in_file(lightning_file,
    'ckpt_dir = "/blob/ICLR/hyena/hyenadna-medium-160k-seqlen"',
    'ckpt_dir = "' + str(hyena_base_model_folder) + '"')
replace_text_in_file(lightning_file,
    'hyenadna_path = "/blob/ICLR/hyena/hyena-dna"',
    'hyenadna_path = "' + str(hyena_code_folder) + '"')

# Switching off the two speed-up options that need compiled FlashAttention add-ons.
replace_text_in_file(lightning_file,
    "            self.model = ConvLMHeadModel(**config)",
    "            config = dict(config)\n"
    "            config[\"fused_dropout_add_ln\"] = False\n"
    "            config[\"fused_mlp\"] = False\n"
    "            self.model = ConvLMHeadModel(**config)")

old_motif_block = """        motifs, bg = scripts.motifs.read_meme(
            cfg.meme_path
        )
        self.tfbs_reward_dict = json.load(open(cfg.tfbs_reward_path, 'r'))
        sel = scripts.utils.load_csv(
            cfg.ppms_path
        ).Matrix_id.tolist()
        self.motif2idx = {name: i for i, name in enumerate(sel)}
        self.idx2motif = {i: name for i, name in enumerate(sel)}
        motifs = [m for m in motifs if m.name.decode() in sel]
        self.motifs = motifs
        self.bg = bg
"""
new_motif_block = """        if cfg.tfbs_lambda > 0:
            motifs, bg = scripts.motifs.read_meme(
                cfg.meme_path
            )
            self.tfbs_reward_dict = json.load(open(cfg.tfbs_reward_path, 'r'))
            sel = scripts.utils.load_csv(
                cfg.ppms_path
            ).Matrix_id.tolist()
            self.motif2idx = {name: i for i, name in enumerate(sel)}
            self.idx2motif = {i: name for i, name in enumerate(sel)}
            motifs = [m for m in motifs if m.name.decode() in sel]
            self.motifs = motifs
            self.bg = bg
        else:
            # PATCH: with a motif weight of 0 the motif reward is never used, so the motif files are not loaded.
            self.tfbs_reward_dict = {}
            self.motif2idx = {}
            self.idx2motif = {}
            self.motifs = []
            self.bg = None
"""
replace_text_in_file(optimizer_file, old_motif_block, new_motif_block)

# Copying the starting sequences from Step 7 to the place where TACO reads them.
starting_sequences_target_folder = taco_folder / "data" / "complex"
starting_sequences_target_folder.mkdir(parents=True, exist_ok=True)
shutil.copy(starting_sequences_file, starting_sequences_target_folder / "mbo.csv")
print("Copied:", starting_sequences_target_folder / "mbo.csv")

### **Step 12: Tiny test run**

TACO is started with the smallest settings (2 rounds, a batch of 8), with the motif weight at 0 and the seed 0. Results go to the folder `results/tiny_test` in Drive. One helper function starts TACO; the same function runs the full runs in Step 15.

In [ ]:
# HELPER FUNCTION
# Starting TACO with the Python of taco_env. Only the last lines of its output are shown, without warnings.
# TACO saves its result files only after the last round, and the file name contains the seed.
def run_taco(seed, rounds, batch_size, results_folder):
    results_folder.mkdir(parents=True, exist_ok=True)
    command = (
        "cd " + str(taco_folder)
        + " && PYTHONPATH=" + str(taco_folder) + ":" + str(taco_folder / "reglm_src") + ":" + str(hyena_code_folder)
        + " " + str(environment_python) + " reinforce_mbo.py --task complex --level mbo"
        + " --seed " + str(seed) + " --tfbs_lambda 0.0 --max_iter " + str(rounds)
        + " --batch_size " + str(batch_size) + " --save_dir " + str(results_folder)
        + " 2>&1 | grep -v Warning | tail -20"
    )
    completed_process = subprocess.run(command, shell=True, capture_output=True, text=True)
    print(completed_process.stdout)
    # 0.0 is the motif weight and 0.0001 is TACO's default learning rate; both are part of the file name.
    result_file = results_folder / ("mbo_0.0_0.0001_complex_mbo_" + str(seed) + ".csv")
    print("Result file saved:", result_file.exists())
    return result_file.exists()

In [ ]:
%%time
run_taco(seeds[0], tiny_test_rounds, tiny_test_batch_size, drive_dir_name / "results" / "tiny_test")

### **Step 13: The full runs, one for each seed**

TACO runs for 100 rounds with a batch of 256 sequences, once for each seed (0, 1 and 42). TACO saves its results only after the last round, so a session that ends in the middle of a run loses that run only.

Reinforcement learning varies a lot between runs. The same run with different seeds shows how much of a result is only luck of the start. With three seeds the spread is a rough sign only.

In [ ]:
%%time
full_run_results_folder = drive_dir_name / "results" / "full_run_100_rounds"
full_run_results_folder.mkdir(parents=True, exist_ok=True)

for seed in seeds:
    print("Seed", seed, ": starting.")
    run_taco(seed, full_run_rounds, full_run_batch_size, full_run_results_folder)

### **Step 14: Summary tables and package versions**

The summary table of every seed is read, and every tenth round is printed. Then the mean and the standard deviation over the seeds are printed for the last round. Finally, the versions of both environments are saved in Drive: `results/00_requirements_colab.txt` (the notebook's own Python on Colab) and `results/00_requirements_taco.txt` (every package in `taco_env`, with exact versions).

Each round, TACO takes the 128 sequences with the best reward-model score. `top` is the mean score of the best 16 of them and `fitness` is the median of the 128 (both from the reward model). `top_oracle` and `fitness_oracle` are the same two numbers from the checker model. `diversity` is the median distance between the 128 sequences, and `novelty` is the median distance of each to its nearest weak starting sequence.

The standard deviation uses only three seeds and only reflects different random starts. The two version files are what is needed to build the same environments again.

In [ ]:
full_run_results_folder = drive_dir_name / "results" / "full_run_100_rounds"
score_columns = ["top", "fitness", "top_oracle", "fitness_oracle", "diversity", "novelty"]

seed_tables = []
for seed in seeds:
    summary_file = full_run_results_folder / ("mbo_0.0_0.0001_complex_mbo_" + str(seed) + "_summary.csv")
    if not summary_file.exists():
        print("Missing:", summary_file.name)
        continue
    seed_table = pandas.read_csv(summary_file)
    seed_table.insert(0, "seed", seed)
    seed_tables.append(seed_table)

all_seeds_table = pandas.concat(seed_tables, ignore_index=True)
print(all_seeds_table[all_seeds_table["round"] % 10 == 0][["seed", "round"] + score_columns].round(3).to_string(index=False))

# Mean and standard deviation over the seeds, for the last round.
last_round_table = all_seeds_table[all_seeds_table["round"] == full_run_rounds][score_columns]
print()
print("Last round, over", len(last_round_table), "seeds:")
print(last_round_table.agg(["mean", "std"]).round(3).to_string())

**References**

1. Yang Z, Su B, Cao J, Wen JR. TACO. ICLR 2025. arXiv:2503.07981. *[The title and the page details are to be checked against the source before this entry is finished.]* Code: https://github.com/yangzhao1230/TACO (MIT license).

The other sources used here (the regLM yeast files, the HyenaDNA code, the yeast promoter data) are listed in the README after their details are checked.